# CARec training

- Load saved splits and cached item embeddings. Preparation stays in [data_exploration.ipynb](data_exploration.ipynb), encoding in [carec_encoder_comparison.ipynb](carec_encoder_comparison.ipynb).
- Phase 1 learns user vectors. Phase 2 freezes that table and trains the item MLP. Cached text vectors stay fixed.
- Select checkpoints using validation NDCG@10, then evaluate warm/cold test once. Each run has its own directory.


## 1. Settings and imports

- Select embeddings, device, and training parameters here. Default to MPS; use `device_name = "cpu"` to switch back.
- Allow up to 200 epochs per phase, with patience 30. Set both epoch limits to 1 for a short end-to-end check.
- Recreate the model after changing settings. Learning rate 0.001 and zero optimizer weight decay follow the released configuration, without a hyperparameter search.


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "requirements.txt").is_file()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from benchmarks.carec import load_carec
from recbole.config import Config
from recbole.data.dataset import Dataset
from recbole.utils import init_seed
from recbole.utils.enum_type import FeatureSource, FeatureType

carec = load_carec()

DATASET_ID = "electronics_carec_035_seed2023"
PREPARED_DIR = PROJECT_ROOT / "data/processed" / DATASET_ID
RECBOLE_ROOT = PROJECT_ROOT / "artifacts/recbole"
RECBOLE_DIR = RECBOLE_ROOT / DATASET_ID
EMBEDDING_NAME = "all-MiniLM-L6-v2.npz"
EMBEDDING_PATH = PROJECT_ROOT / "artifacts/embeddings" / DATASET_ID / EMBEDDING_NAME

seed = 2023
phase1_epochs = 20
phase2_epochs = 20
patience = 10
weight_decay = 0.0
training_batch_size = 1024
learning_rate = 0.001

device_name = "mps"
if device_name == "mps" and not torch.backends.mps.is_available():
    raise RuntimeError("MPS is unavailable in this kernel. Select the project environment or use CPU.")
device = torch.device(device_name)


/Users/matouskovar/FIT/NI-PML/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Load data with RecBole

- RecBole reads the saved splits and assigns integer IDs. It reserves ID 0 for padding.
- Export interaction files only when missing. All embedding variants use these same files.
- All splits share ID mappings. Only training interactions will enter the graph.


In [2]:
with np.load(EMBEDDING_PATH, allow_pickle=False) as saved:
    embedding_item_ids = saved["item_ids"]
    item_embeddings = saved["embeddings"]
    embedding_model_id = str(saved["model_id"])
    embedding_prompt = str(saved["prompt"])
    embedding_revision = str(saved["revision"])

RECBOLE_DIR.mkdir(parents=True, exist_ok=True)
interaction_columns = ["user_id:token", "item_id:token", "timestamp:float"]
for split_name in ("train", "valid", "test", "cold"):
    split_path = RECBOLE_DIR / f"{DATASET_ID}.{split_name}.inter"
    if not split_path.exists():
        interactions = pd.read_csv(
            PREPARED_DIR / f"{split_name}.tsv.gz",
            sep="\t",
            usecols=interaction_columns,
            dtype={"user_id:token": str, "item_id:token": str},
            float_precision="round_trip",
        )
        # RecBole requires typed headers. Leave source_row out of this export.
        interactions[interaction_columns].to_csv(split_path, sep="\t", index=False)

catalogue_path = RECBOLE_DIR / f"{DATASET_ID}.item"
if not catalogue_path.exists():
    pd.DataFrame({"item_id:token": embedding_item_ids}).to_csv(
        catalogue_path, sep="\t", index=False,
    )

print(f"{embedding_model_id}: {item_embeddings.shape}, prompt={embedding_prompt!r}")


sentence-transformers/all-MiniLM-L6-v2: (34220, 384), prompt=''


In [3]:
dataset_settings = {
    "data_path": str(RECBOLE_ROOT),
    "benchmark_filename": ["train", "valid", "test", "cold"],
    "field_separator": "\t",
    "USER_ID_FIELD": "user_id",
    "ITEM_ID_FIELD": "item_id",
    "TIME_FIELD": "timestamp",
    "load_col": {
        "inter": ["user_id", "item_id", "timestamp"],
        "item": ["item_id"],
    },
    "normalize_all": False,
    "use_gpu": False,  # RecBole uses this flag for CUDA, not MPS.
}

config = Config(model=carec.CollabContex, dataset=DATASET_ID, config_dict=dataset_settings)
# Set before construction: CARec stores graph and item tensors outside registered buffers.
config["device"] = device
recbole_dataset = Dataset(config)
recbole_product_ids = recbole_dataset.field2id_token["item_id"]


/Users/matouskovar/FIT/NI-PML/.venv/lib/python3.11/site-packages/recbole/data/dataset/dataset.py:638: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  feat[field].fillna(value=0, inplace=True)
/Users/matouskovar/FIT/NI-PML/.venv/lib/python3.11/site-packages/recbole/data/dataset/dataset.py:640: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on

## 3. Attach embeddings and build splits

- Cache rows and RecBole IDs can have different orders. Match them by original product ID and leave padding as zeros.
- Register the vectors as `item_emb`, the feature expected by CARec.
- `build()` converts tables to tensors and separates the saved splits. Rerun section 2 before repeating this section.


In [4]:
embedding_row_by_product = {}
for embedding_row, product_id in enumerate(embedding_item_ids):
    embedding_row_by_product[product_id] = embedding_row

assert len(embedding_row_by_product) == len(embedding_item_ids), "Duplicate product IDs in cache"
assert recbole_product_ids[0] == "[PAD]"
assert set(recbole_product_ids[1:]) == set(embedding_item_ids), "Product catalogues differ"

embedding_dimension = item_embeddings.shape[1]
recbole_item_embeddings = np.zeros(
    (recbole_dataset.item_num, embedding_dimension), dtype=np.float32,
)
# Row number must equal the RecBole product ID, not the original cache row.
for recbole_id in range(1, recbole_dataset.item_num):
    product_id = recbole_product_ids[recbole_id]
    recbole_item_embeddings[recbole_id] = item_embeddings[embedding_row_by_product[product_id]]

recbole_dataset.set_field_property(
    "item_emb", FeatureType.FLOAT_SEQ, FeatureSource.ITEM, embedding_dimension,
)
recbole_dataset.item_feat["item_emb"] = list(recbole_item_embeddings)
train_dataset, valid_dataset, test_dataset, cold_dataset = recbole_dataset.build()


## 4. Create a fresh CARec model

- Use only training edges. Match model dimensions to the selected embeddings.
- Rerun this section to reset weights before a new run.
- On MPS, uniformity runs on CPU with gradients preserved. Graph propagation and model parameters stay on MPS.


In [5]:
config["seed"] = seed
config["encoder"] = "LightGCN"
config["num_layers"] = 2
config["embedding_size"] = embedding_dimension
config["mlp"] = {
    "input_dim": embedding_dimension,
    "hidden_dims": [embedding_dimension, embedding_dimension],
    "output_dim": embedding_dimension,
    "num_layers": 2,
    "dropout": 0.2,
}

init_seed(config["seed"], config["reproducibility"])
carec_model = carec.CollabContex(config, train_dataset).to(config["device"])

if device.type == "mps":
    # pdist lacks an MPS kernel. CPU transfer preserves autograd and the original loss.
    def mps_uniformity(vectors, t=2):
        return carec.CollabContex.uniformity(vectors.cpu(), t=t).to(vectors.device)

    carec_model.uniformity = mps_uniformity

assert carec_model.encoder.user_embedding.weight.device.type == device.type
assert carec_model.encoder.item_embedding.device.type == device.type
assert carec_model.encoder.edge_index.device.type == device.type
assert carec_model.encoder.edge_weight.device.type == device.type

print(f"Embedding variant: {EMBEDDING_NAME}, dimension: {embedding_dimension}")
print(f"Users: {train_dataset.user_num - 1:,}, products: {train_dataset.item_num - 1:,}")
print({name: len(split.inter_feat) for name, split in zip(
    config["benchmark_filename"], [train_dataset, valid_dataset, test_dataset, cold_dataset],
)})
print(f"Directed graph entries: {carec_model.encoder.edge_index.shape[1]:,}")
print(f"Device: {config['device']}")


Embedding variant: all-MiniLM-L6-v2.npz, dimension: 384
Users: 81,520, products: 34,220
{'train': 452606, 'valid': 14853, 'test': 133274, 'cold': 23163}
Directed graph entries: 905,212
Device: mps


## 5. Evaluation helpers

- RecBole computes per-user Recall/NDCG at 10 and 50. Warm ranks the full catalogue; cold ranks cold products only and bypasses MLP. [Paper, section 4.1.3](https://yangliangwei.github.io/publication/cikm2024_carec/CIKM2024_CARec.pdf).
- Mask padding and earlier history, except current positives. Repeated pairs across splits remain an unresolved benchmark limitation.
- Validation selects checkpoints. Test scores never select epochs or settings.


In [6]:
from recbole.data.dataloader import FullSortEvalDataLoader
from recbole.sampler import Sampler
from recbole.evaluator import Collector, Evaluator

config["metrics"] = ["Recall", "NDCG"]
config["topk"] = [10, 50]
# RecBole measures eval_batch_size in user-product scores, not users.
config["eval_batch_size"] = 64 * train_dataset.item_num

# Upstream repeatable sampling leaves history empty. Supply cumulative history.
history_sampler = Sampler(
    ["train", "valid", "test", "cold"],
    [train_dataset, valid_dataset, test_dataset, cold_dataset],
)
evaluation_loaders = {}
for split_name, split in [("valid", valid_dataset), ("warm", test_dataset), ("cold", cold_dataset)]:
    phase = "test" if split_name == "warm" else split_name
    evaluation_loaders[split_name] = FullSortEvalDataLoader(
        config, split, history_sampler.set_phase(phase), shuffle=False,
    )

cold_product_ids = cold_dataset.inter_feat[config["ITEM_ID_FIELD"]].unique()
train_product_ids = train_dataset.inter_feat[config["ITEM_ID_FIELD"]].unique()
assert not set(cold_product_ids.tolist()) & set(train_product_ids.tolist())
print({name: len(loader.uid_list) for name, loader in evaluation_loaders.items()})


{'valid': 12386, 'warm': 81520, 'cold': 18680}


In [7]:
@torch.no_grad()
def evaluate_carec(model, loader, cold_only=False):
    collector = Collector(config)
    item_count = loader.dataset.item_num
    evaluator = Evaluator(config)
    previous_training = model.training
    previous_mlp_bypass = model.encoder.not_apply_mlp
    model.eval()
    if cold_only:
        model.encoder.not_apply_mlp = True
    # full_sort_predict caches graph outputs. Never reuse them across modes or epochs.
    model.restore_user_e = model.restore_item_e = None
    candidate_mask = torch.ones(item_count, dtype=torch.bool, device=config["device"])
    if cold_only:
        candidate_mask[:] = False
        cold_ids = loader.dataset.inter_feat[config["ITEM_ID_FIELD"]].unique()
        candidate_mask[cold_ids.to(config["device"])] = True
    candidate_mask[0] = False

    try:
        for interaction, history, positive_users, positive_items in tqdm(loader, desc="Cold" if cold_only else "Full ranking"):
            scores = model.full_sort_predict(interaction.to(config["device"]))
            scores = scores.view(-1, item_count)
            if not torch.isfinite(scores).all():
                raise ValueError("Non-finite recommendation scores before masking.")
            # Apply the item mask to columns, for every user in the batch.
            scores[:, ~candidate_mask] = -torch.inf
            if history is not None:
                scores[history[0].to(scores.device), history[1].to(scores.device)] = -torch.inf
            collector.eval_batch_collect(scores, interaction, positive_users, positive_items)
        return {name: float(value) for name, value in evaluator.evaluate(collector.get_data_struct()).items()}
    finally:
        model.encoder.not_apply_mlp = previous_mlp_bypass
        model.restore_user_e = model.restore_item_e = None
        model.train(previous_training)


## 6. Run directory and training function

- Save `config.json`, ID mappings, one best checkpoint and an epoch history per phase under `artifacts/carec/`.
- Validate after each epoch. Save when NDCG@10 improves; stop after 30 consecutive epochs without improvement.
- Each phase gets a fresh Adam optimizer and stopping counter. Reload its best weights before continuing.
- Checkpoints include optimizer state. Item vectors and graph are rebuilt from the same saved cache and splits; automatic interrupted-run resume is not implemented.


In [8]:
import json
from datetime import datetime, timezone
from time import perf_counter
from uuid import uuid4

run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid4().hex[:6]
RUN_DIR = PROJECT_ROOT / "artifacts/carec" / DATASET_ID / EMBEDDING_PATH.stem / run_id
RUN_DIR.mkdir(parents=True, exist_ok=False)

run_settings = {
    "dataset": DATASET_ID, "embedding_file": str(EMBEDDING_PATH.relative_to(PROJECT_ROOT)),
    "encoder": embedding_model_id, "revision": embedding_revision, "prompt": embedding_prompt,
    "carec_commit": carec.commit, "seed": seed, "device": str(device),
    "phase1_epochs": phase1_epochs, "phase2_epochs": phase2_epochs, "patience": patience,
    "learning_rate": learning_rate, "weight_decay": weight_decay, "batch_size": training_batch_size,
    "dimension": embedding_dimension, "num_layers": config["num_layers"], "mlp": config["mlp"],
    "metrics": config["metrics"], "topk": config["topk"], "selection_metric": "ndcg@10",
    "history_mask": "cumulative history minus current positives",
    "warm_candidates": "full catalogue", "cold_candidates": "cold-split products only",
    "torch_version": str(torch.__version__),
}

def write_json(path, data):
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(data, indent=2, allow_nan=False) + "\n")
    temporary.replace(path)

write_json(RUN_DIR / "config.json", run_settings)
np.savez(
    RUN_DIR / "id_maps.npz",
    user_ids=np.asarray(train_dataset.field2id_token["user_id"], dtype=str),
    item_ids=np.asarray(train_dataset.field2id_token["item_id"], dtype=str),
)
print(f"Run directory: {RUN_DIR}")


Run directory: /Users/matouskovar/FIT/NI-PML/artifacts/carec/electronics_carec_035_seed2023/all-MiniLM-L6-v2/20261105T095720Z-d2cb19


In [9]:
def train_phase(model, phase, max_epochs):
    if max_epochs < 1 or patience < 1:
        raise ValueError("Epoch limit and patience must be positive.")
    if phase == 1:
        model.set_item_tutoring_phase()
        uniformity_field = config["USER_ID_FIELD"]
    elif phase == 2:
        model.set_user_tutoring_phase()
        uniformity_field = config["ITEM_ID_FIELD"]
    else:
        raise ValueError("CARec has phases 1 and 2.")

    # Frozen parameters have no optimizer state in this phase.
    model.zero_grad(set_to_none=True)
    optimizer = torch.optim.Adam(
        [parameter for parameter in model.parameters() if parameter.requires_grad],
        lr=learning_rate, weight_decay=weight_decay,
    )
    generator = torch.Generator().manual_seed(seed + phase)
    checkpoint_path = RUN_DIR / f"phase{phase}-best.pt"
    best_score, stale_epochs = -float("inf"), 0
    history = []
    started = perf_counter()

    for epoch in range(1, max_epochs + 1):
        model.train()
        indices = torch.randperm(len(train_dataset.inter_feat), generator=generator)
        total_loss, interaction_count = 0.0, 0
        for batch_ids in tqdm(indices.split(training_batch_size), desc=f"Phase {phase}, epoch {epoch}"):
            batch = train_dataset.inter_feat[batch_ids].to(config["device"])
            if batch[uniformity_field].unique().numel() < 2:
                raise ValueError("Uniformity needs at least two distinct active-side IDs.")
            optimizer.zero_grad(set_to_none=True)
            loss = model.calculate_loss(batch)
            if not torch.isfinite(loss):
                raise ValueError("Training loss is not finite.")
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(batch)
            interaction_count += len(batch)

        valid_metrics = evaluate_carec(model, evaluation_loaders["valid"])
        score = float(valid_metrics["ndcg@10"])
        if not np.isfinite(score):
            raise ValueError("Validation NDCG@10 is not finite.")
        improved = score > best_score
        stale_epochs = 0 if improved else stale_epochs + 1
        history.append({
            "epoch": epoch, "loss": total_loss / interaction_count,
            "valid": valid_metrics, "elapsed_seconds": perf_counter() - started,
        })
        if improved:
            best_score = score
            # Write a temporary file so interruption cannot truncate the previous best.
            temporary = checkpoint_path.with_suffix(".pt.tmp")
            torch.save({
                "state_dict": {name: value.detach().cpu() for name, value in model.state_dict().items()},
                "optimizer": optimizer.state_dict(), "phase": phase, "epoch": epoch,
                "valid": valid_metrics, "settings": run_settings,
            }, temporary)
            temporary.replace(checkpoint_path)
        write_json(RUN_DIR / f"phase{phase}-history.json", history)
        print(f"Phase {phase}, epoch {epoch}: loss={history[-1]['loss']:.4f}, valid NDCG@10={score:.4f}, patience={stale_epochs}/{patience}")
        if stale_epochs >= patience:
            break

    # PyTorch 2.6 defaults to restricted loading. These checkpoints contain tensors and plain data.
    best = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
    model.load_state_dict(best["state_dict"])
    model.restore_user_e = model.restore_item_e = None
    model.eval()
    return {"best_epoch": best["epoch"], "valid": best["valid"], "epochs_run": len(history)}


## 7. Train both phases

- Phase 2 starts from the best phase-one weights. MLP is active; the user embedding table is frozen.
- Run section 4 first for a fresh model, and section 6 for a new output directory. Rerunning these calls continues current weights with fresh optimizers.
- Each phase can run up to its configured limit. Both use validation NDCG@10 for early stopping.


In [ ]:
phase_results = {}
phase_results["phase1"] = train_phase(carec_model, phase=1, max_epochs=phase1_epochs)
phase_results["phase2"] = train_phase(carec_model, phase=2, max_epochs=phase2_epochs)
write_json(RUN_DIR / "training.json", phase_results)
display(pd.DataFrame(phase_results).T)


Phase 1, epoch 1:  29%|██▉       | 129/442 [02:47<06:37,  1.27s/it]

## 8. Evaluate and save final results

- Use the best phase-two checkpoint already loaded above. Warm uses MLP; cold temporarily bypasses it and recomputes graph vectors.
- Save warm/cold metrics to `results.json`. Keep the embedding cache and saved splits alongside the checkpoints for later reconstruction.


In [ ]:
if carec_model.encoder.not_apply_mlp or "phase2" not in phase_results:
    raise ValueError("Complete both training phases before final evaluation.")

final_results = {}
for split_name in ("warm", "cold"):
    final_results[split_name] = evaluate_carec(
        carec_model, evaluation_loaders[split_name], cold_only=(split_name == "cold"),
    )
write_json(RUN_DIR / "results.json", {
    "training": phase_results, "test": final_results,
    "evaluated_users": {name: len(evaluation_loaders[name].uid_list) for name in final_results},
})
display(pd.DataFrame(final_results).T)
print(f"Saved checkpoints, history, settings, and metrics to {RUN_DIR}")
